# Detailliertes Finanzreporting: Apple Inc. (AAPL)

---

| |                                     |
|---|-------------------------------------|
| **Unternehmen** | Apple Inc. (AAPL)                   |
| **Zeitraum** | 01. Januar 2019 – 31. Dezember 2024 |
| **Datenquelle** | Yahoo Finance (via `yfinance`)      |
---

## 1. Einleitung

### Analysemotivation

Apple Inc. (AAPL) wurde für diese Analyse gewählt, weil das Unternehmen mehrere zentrale Anforderungen an ein geeignetes Analyseobjekt erfüllt:

- **Krisenresilienz:** AAPL hat globale Schocks wie die COVID-19-Pandemie (2020) und den damit verbundenen Börseneinbruch von rund 30 % erfolgreich überstanden und sich anschließend stark erholt – ein idealer Stresstest für Rendite- und Risikoanalysen.
- **Datenverfügbarkeit und -qualität:** Tägliche Handelsdaten über den gesamten Zeitraum 2019–2024 sind vollständig und zuverlässig verfügbar.

## 2. Datenbeschaffung & Aufbereitung
Alle benötigten Bibliotheken werden importiert und die Rohdaten über die Yahoo Finance API geladen.

### 2.1 Bibliotheken
Die Bibliotheken `yfinance` und `plotly` werden für die Datenbeschaffung, -analyse und -manipulation verwendet.

In [1]:
import yfinance as yf
import plotly.express as px

### 2.2 Datenbeschaffung & Aufbereitung
Die Kursdaten werden über die Bibliothek `yfinance` direkt von Yahoo Finance runtergeladen. Der Abruf umfasst den Zeitraum vom 01. Januar 2019 bis zum 31. Dezember 2024 und enthält die täglichen OHLCV-Daten (Open, High, Low, Close, Adjusted Close, Volume). Die Spaltennamen werden zur besseren Lesbarkeit in Kleinbuchstaben umbenannt.

In [2]:
START_DATE = "2019-01-01"
END_DATE = "2024-12-31"

# Download Apple Inc. (AAPL) Daten und Aufbereitung des DataFrames
aapl = (yf.download("AAPL", start=START_DATE, end=END_DATE, auto_adjust=False)
        .reset_index()  # Datum als Spalte statt Index
        .rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjusted",
            "Volume": "volume"})
        )

# MultiIndex-Spalten von yfinance auf einfache Spaltennamen reduzieren
aapl.columns = [col[0] for col in aapl.columns]

aapl

[*********************100%***********************]  1 of 1 completed


,date,adjusted,close,high,low,open,volume
0,2019-01-02,37.503723,39.480000,39.712502,38.557499,38.722500,148158800
1,2019-01-03,33.768078,35.547501,36.430000,35.500000,35.994999,365248800
2,2019-01-04,35.209625,37.064999,37.137501,35.950001,36.132500,234428400
3,2019-01-07,35.131245,36.982498,37.207500,36.474998,37.174999,219111200
4,2019-01-08,35.800968,37.687500,37.955002,37.130001,37.389999,164101200
...,...,...,...,...,...,...,...
1504,2024-12-23,253.883118,255.270004,255.649994,253.449997,254.770004,40858800
1505,2024-12-24,256.797211,258.200012,258.209991,255.289993,255.490005,23234700
1506,2024-12-26,257.612701,259.019989,260.100006,257.630005,258.190002,27237100
1507,2024-12-27,254.201385,255.589996,258.700012,253.059998,257.829987,42355300


## 3. Datenqualität & -bereinigung

Vor der eigentlichen Analyse wird der Datensatz auf mögliche Qualitätsprobleme geprüft. Konkret wird der Schlusskurs (`close`) auf drei häufige Datenfehler untersucht:

- **Fehlende Werte (`NaN`):** Tage ohne Kursangabe, z. B. durch API-Fehler oder Handelsunterbrechungen.
- **Nullwerte:** Ein Kurs von 0 ist ökonomisch nicht plausibel und deutet auf einen Datenfehler hin.
- **Negative Werte:** Aktienkurse können definitionsgemäß nicht negativ sein.

In [3]:
issues_before = {
    'missing': aapl['close'].isna().sum().item(), # item() extrahiert den skalaren Wert
    'zeros': (aapl['close'] == 0).sum().item(),
    'negatives': (aapl['close'] < 0).sum().item()
}

In [4]:
print(f"Fehlende Werte: {issues_before['missing']}")
print(f"Nullwerte: {issues_before['zeros']}")
print(f"Negative Werte: {issues_before['negatives']}")

Fehlende Werte: 0
Nullwerte: 0
Negative Werte: 0


Das Ergebnis zeigt, dass der Datensatz keine dieser Anomalien enthält – die Daten sind vollständig und konsistent und können ohne weitere Bereinigung verwendet werden.

---

**Quelle:**
- [Data Cleaning & Validation in Financial Modeling – cinderzhang.github.io](https://cinderzhang.github.io/DRIVER-FinancialModeling/Session3-Data-Cleaning-Validation.html)

## 4. Detaillierte Analysen

### 4.1 Kursverlauf
Das folgende Diagramm zeigt den adjustierten Schlusskurs der Apple-Aktie im Zeitraum 2019–2024. Der adjustierte Kurs berücksichtigt Dividendenzahlungen und Aktiensplits, wodurch die langfristige Kursentwicklung vergleichbar bleibt.

Deutlich erkennbar ist der starke Einbruch im März 2020 infolge der COVID-19-Pandemie, gefolgt von einer rasanten Erholung. In den darauffolgenden Jahren stieg der Kurs kontinuierlich an und erreichte Ende 2024 ein Niveau von rund 250 USD – mehr als das Dreifache des Wertes zu Beginn des Betrachtungszeitraums.

In [5]:
adjusted_line_chart = px.line(
    aapl,
    x="date",
    y="adjusted",
    title="Apple Inc. (AAPL) – Adjustierter Schlusskurs 2019–2024",
    labels={"date": "Datum", "adjusted": "Kurs (USD)"}
)

adjusted_line_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Kurs (USD)",
    hovermode="x unified"
)

adjusted_line_chart.show()

Der Kursverlauf zeigt deutlich drei Phasen: einen starken Einbruch im März 2020 (COVID-Crash), gefolgt von einer rasanten Erholung und einem kontinuierlichen Anstieg bis Ende 2021. Ab Januar 2022 folgte eine Korrekturphase, die bis Januar 2023 andauerte. Danach erholte sich die Aktie erneut und erreichte Ende 2024 ein Kursniveau von rund **250 USD** – mehr als das **Dreifache** des Startwerts von ~80 USD im Januar 2019.

### 4.2 Renditeanalyse

Die täglichen Renditen bilden die **Grundlage für alle weiteren Analysen** in diesem Bericht – insbesondere für die Volatilitätsanalyse, die Risikoberechnung (Value at Risk) und den Vergleich mit der Benchmark. Die einfache Rendite ist definiert als:

$$r_t = p_t / p_{t-1} - 1$$

wobei $p_t$ der adjustierte Schlusskurs am Tag $t$ und $p_{t-1}$ der Kurs am Vortag ist.

In [6]:
returns = (aapl
           .sort_values('date')
           .assign(ret=lambda x: x['adjusted'].pct_change())
           .dropna() # Entfernt die erste Zeile (NaN), da dort die Rendite nicht berechnet werden kann (kein Vortag)
           [['date', 'ret']]
           )

#### Zeitreihe der täglichen Renditen

Das folgende Diagramm zeigt die täglichen Renditen im Zeitverlauf. Phasen mit hohen Ausschlägen nach oben oder unten deuten auf erhöhte Volatilität hin – besonders deutlich erkennbar ist der COVID-Schock im März 2020, der die stärksten Kursbewegungen des gesamten Betrachtungszeitraums verursachte.

In [7]:
returns_line_chart = px.line(
    returns,
    x="date",
    y="ret",
    title="Apple Inc. (AAPL) – Tägliche Renditen 2019–2024",
    labels={"date": "Datum", "ret": "Rendite"}
)

returns_line_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Rendite",
    yaxis_tickformat=".1%",
    hovermode="x unified"
)

returns_line_chart.show()

Die Zeitreihe zeigt, dass die täglichen Renditen überwiegend in einem engen Band zwischen **–2 % und +2 %** liegen. Die mit Abstand stärksten Ausschläge traten im **März 2020** auf – sowohl nach unten (Panikverkäufe) als auch nach oben (Erholungsbewegungen). Abseits dieser Krisenphase ist das Bild gleichmäßig verteilt um die Nulllinie, was auf keinen systematischen Trend in den täglichen Schwankungen hindeutet.

#### Verteilung der täglichen Renditen

Das Histogramm zeigt, wie häufig bestimmte Renditebereiche aufgetreten sind. Eine enge, symmetrische Verteilung um null deutet auf stabiles Marktverhalten hin. Breite Ausläufer (sogenannte *Fat Tails* oder Leptokurtosis) links und rechts signalisieren, dass extreme Tagesrenditen häufiger auftreten als eine Normalverteilung erwarten würde. Dies ist ein empirisch gut belegtes Merkmal von Aktienrenditen und hat direkte Auswirkungen auf Risikomodelle wie den Value at Risk: Modelle, die Normalverteilung annehmen, unterschätzen systematisch die Wahrscheinlichkeit extremer Verluste.

---

**Quellen:**
- [Fat-tailed distribution – Wikipedia](https://en.wikipedia.org/wiki/Fat-tailed_distribution)
- [Leptokurtic Fat Tails: Understanding the Impact of Extreme Outliers – FasterCapital](https://fastercapital.com/content/Leptokurtic-Fat-Tails--Understanding-the-Impact-of-Extreme-Outliers.html)
- [COVID-19 and the March 2020 Stock Market Crash – PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC7343658/)

In [8]:
returns_histogram = px.histogram(
    returns,
    x="ret",
    nbins=100,
    title="Apple Inc. (AAPL) – Verteilung der täglichen Renditen 2019–2024",
    labels={"ret": "Rendite"}
)

returns_histogram.update_layout(
    xaxis_title="Rendite",
    yaxis_title="Häufigkeit",
    xaxis_tickformat=".1%",
    bargap=0.05
)

returns_histogram.show()

Das Histogramm bestätigt die theoretische Erwartung: Die Verteilung ist **glockenförmig und um null zentriert**, weist jedoch erkennbar breitere Ränder auf als eine Normalverteilung. Die meisten Renditen liegen zwischen –2 % und +2 %, aber einzelne Extremtage mit Renditen jenseits von ±5 % treten häufiger auf als ein Normalverteilungsmodell vorhersagen würde – ein klassisches Zeichen für **Leptokurtosis** (Fat Tails) in Aktienrenditen.

### 4.3 Volatilitätsanalyse

Volatilität ist eine Kennzahl, mit der die **Schwankungsbreite der Kurse** von Finanzinstrumenten gemessen wird. Sie gilt als zentrales Risikomaß in der Finanzanalyse: Eine hohe Volatilität bedeutet größere Unsicherheit und stärkere Kursschwankungen – sowohl nach unten als auch nach oben.

Berechnet wird die Volatilität als **annualisierte Standardabweichung** der täglichen Renditen:

$$\sigma = \text{std}(r_t) \times \sqrt{252}$$

wobei $\sqrt{252}$ die Anzahl der Handelstage pro Jahr annualisiert. Der Prozess umfasst vier Schritte:

1. Berechnung der durchschnittlichen täglichen Rendite
2. Bestimmung der Varianz aus den Abweichungen vom Mittelwert
3. Ziehen der Wurzel (= Standardabweichung)
4. Annualisierung durch Multiplikation mit $\sqrt{252}$

Eine niedrige Volatilität deutet auf stabile, besser kalkulierbare Kursentwicklungen hin – eine hohe Volatilität erschwert die Vorhersagbarkeit und erhöht das Verlustrisiko. Dabei gilt das fundamentale Börsenprinzip: **Ohne Risiko keine Rendite** – höhere Volatilität geht in der Regel mit höheren Renditechancen einher.

---

**Quelle:**
- [Volatilität: Standardabweichung zur Risikobewertung von Aktien – gevestor.de](https://www.gevestor.de/finanzwissen/boerse/anlagenanalyse/volatilitaet-standardabweichung-zur-risikobewertung-von-aktien.html)

#### 4.3.1 Berechnung der Volatilität mit `.std()`

In Python berechnet `pandas` die Volatilität in zwei Schritten:

**Schritt 1 – Tägliche Standardabweichung**

```python
returns['ret'].std()
```

`.std()` übernimmt dabei automatisch die gesamte Varianzberechnung:
1. Mittelwert $\bar{r}$ aller täglichen Renditen ermitteln
2. Für jeden Tag die Abweichung $(r_t - \bar{r})$ berechnen und quadrieren
3. Summe der quadrierten Abweichungen durch $N-1$ teilen → Varianz $s^2$
4. Wurzel der Varianz ziehen → tägliche Standardabweichung $s$

Der Nenner $N-1$ (statt $N$) ist die sogenannte **Bessel-Korrektur**: Da wir nur eine Stichprobe der möglichen Handelstage beobachten, korrigiert $N-1$ den Schätzer, sodass er die wahre Streuung der Grundgesamtheit nicht unterschätzt.

**Schritt 2 – Annualisierung**

```python
annual_vol = daily_vol * (252 ** 0.5)
```

Da die Standardabweichung mit der Wurzel der Zeit skaliert, multipliziert man die tägliche Volatilität mit $\sqrt{252}$ (Anzahl der Handelstage pro Jahr). Das Ergebnis `annual_vol` ist die **annualisierte Volatilität** – sie gibt an, wie stark der Kurs der Aktie im Durchschnitt über ein gesamtes Jahr schwankt. Ein Wert von z. B. 0,28 bedeutet, dass AAPL jährlich um rund 28 % schwankt.

#### 4.3.2 Rollierende Volatilität

Ein einzelner Volatilitätswert über den gesamten Zeitraum verbirgt, dass Märkte nicht konstant schwanken. Die **rollierende Volatilität** berechnet die annualisierte Standardabweichung in einem gleitenden 30-Tage-Fenster und macht dadurch sichtbar, wann Ruhephasen und wann turbulente Phasen aufgetreten sind.

Ein starker Anstieg der rollierenden Volatilität ist ein klassisches Signal für Marktstress – wie etwa der COVID-Schock im März 2020.

---

**Quelle:**
- [Volatilität durch rollierende Renditen verstehen – FasterCapital](https://fastercapital.com/de/inhalt/Volatilitaet-durch-rollierende-Renditen-verstehen.html)

In [9]:
TRADING_DAYS_PER_YEAR = 252
DAYS = 30
rolling_vol = (returns
               .assign(rolling_vol=lambda x: x['ret'].rolling(DAYS).std() * (TRADING_DAYS_PER_YEAR ** 0.5))
               .dropna(subset=['rolling_vol'])
               )

rolling_vol_chart = px.line(
    rolling_vol,
    x="date",
    y="rolling_vol",
    title="Apple Inc. (AAPL) – Rollierende Volatilität (30 Tage) 2019–2024",
    labels={"date": "Datum", "rolling_vol": "Volatilität (annualisiert)"}
)

rolling_vol_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Volatilität (annualisiert)",
    yaxis_tickformat=".1%",
    hovermode="x unified"
)

rolling_vol_chart.show()

Das Diagramm zeigt deutlich, dass die Volatilität nicht konstant ist, sondern in kurzen, intensiven Stressphasen stark ansteigt. Der **COVID-Crash im März 2020** erzeugte einen markanten Spike auf über **80 % annualisierte Volatilität** – das Maximum des gesamten Zeitraums. In der **Zinsanstiegsphase 2022** war ebenfalls eine erhöhte Volatilität erkennbar, die sich aber über einen längeren Zeitraum verteilte. In ruhigen Marktphasen bewegt sich die rollierende Volatilität von AAPL typischerweise zwischen **15 % und 30 %**.

In [10]:
avg_rolling_vol = rolling_vol['rolling_vol'].mean()
print(f"Durchschnittliche rollierende Volatilität (30 Tage, annualisiert): {avg_rolling_vol:.2%}")

Durchschnittliche rollierende Volatilität (30 Tage, annualisiert): 28.08%


### 4.4 Sharpe-Ratio


Die Sharpe Ratio ist eine 1966 von Nobelpreisträger William F. Sharpe entwickelte Kennzahl, die die **Performance einer Anlage in Relation zum eingegangenen Risiko** bewertet. Sie beantwortet die Frage: Wie viel Rendite erhalte ich pro Einheit Risiko?

Die Formel lautet:

$$SR = \frac{r_p - r_f}{\sigma_p}$$

| Variable | Bedeutung |
|---|---|
| $r_p$ | Annualisierte Rendite des Portfolios / der Aktie |
| $r_f$ | Risikoloser Zinssatz (z. B. Geldmarktrendite) |
| $\sigma_p$ | Annualisierte Volatilität (Standardabweichung der Renditen) |

**Interpretation:**

| Wert | Bedeutung |
|---|---|
| $> 1$ | Hervorragende Entschädigung für das eingegangene Risiko |
| $= 1$ | Chancen und Risiken im ausgewogenen Verhältnis |
| $< 1$ | Unterdurchschnittliche Risikoentschädigung |
| $\approx 0$ | Rendite entspricht etwa dem risikolosen Zins |
| $< 0$ | Verlust oder Unterperformance gegenüber dem risikolosen Zins |

---

**Quelle:**
- [Sharpe Ratio – Fidelity Deutschland](https://www.fidelity.de/wissen/tipps-and-strategien/risiko-kennziffern/sharpe-ratio/)

In [11]:
RISK_FREE_RATE = 0.02  # 2% risikoloser Zinssatz (Näherungswert für den Betrachtungszeitraum 2019–2024)

annual_return = returns['ret'].mean() * TRADING_DAYS_PER_YEAR
annual_vol = returns['ret'].std() * (TRADING_DAYS_PER_YEAR ** 0.5)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_vol

print(f"Annualisierte Rendite:  {annual_return:.2%}")
print(f"Annualisierte Vola:    {annual_vol:.2%}")
print(f"Sharpe Ratio (gesamt): {sharpe_ratio:.2f}")

Annualisierte Rendite:  36.53%
Annualisierte Vola:    30.85%
Sharpe Ratio (gesamt): 1.12


#### 4.4.1 Rollierende Sharpe Ratio

Ein einzelner Sharpe-Ratio-Wert über den gesamten Zeitraum verbirgt, dass sich die risikobereinigte Performance einer Aktie stark verändert. Die **rollierende Sharpe Ratio** berechnet die Kennzahl in einem gleitenden 252-Tage-Fenster (= 1 Handelsjahr) und macht dadurch sichtbar, wann AAPL Anleger gut für das eingegangene Risiko entschädigt hat – und wann nicht.

Zwei Referenzlinien helfen bei der Einordnung:
- **SR = 1** (grün gestrichelt): Ab hier gilt die risikobereinigte Performance als gut
- **SR = 0** (rot gestrichelt): Darunter lag die Rendite unter dem risikolosen Zins

---

**Quellen:**
- [Sharpe Ratio – Fidelity Deutschland](https://www.fidelity.de/wissen/tipps-and-strategien/risiko-kennziffern/sharpe-ratio/)
- [Rolling Sharpe Ratio – QuantLib / QuantStats Dokumentation](https://quantstats.readthedocs.io/en/latest/)
- [Sharpe Ratio: Definition and Formula – Investopedia](https://www.investopedia.com/terms/s/sharperatio.asp)

In [12]:
ROLLING_WINDOW = 252  # 1 Handelsjahr

rolling_sharpe = (
    returns
    .assign(
        rolling_mean=lambda x: x['ret'].rolling(ROLLING_WINDOW).mean() * TRADING_DAYS_PER_YEAR,
        rolling_std=lambda x: x['ret'].rolling(ROLLING_WINDOW).std() * (TRADING_DAYS_PER_YEAR ** 0.5)
    )
    .assign(rolling_sharpe=lambda x: (x['rolling_mean'] - RISK_FREE_RATE) / x['rolling_std'])
    .dropna(subset=['rolling_sharpe'])
    [['date', 'rolling_sharpe']]
)

rolling_sharpe_chart = px.line(
    rolling_sharpe,
    x="date",
    y="rolling_sharpe",
    title="Apple Inc. (AAPL) – Rollierende Sharpe Ratio (252 Tage) 2019–2024",
    labels={"date": "Datum", "rolling_sharpe": "Sharpe Ratio"}
)

rolling_sharpe_chart.add_hline(y=1, line_dash="dash", line_color="green",
                                annotation_text="SR = 1 (gut)", annotation_position="bottom right")
rolling_sharpe_chart.add_hline(y=0, line_dash="dash", line_color="red",
                                annotation_text="SR = 0", annotation_position="bottom right")

rolling_sharpe_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Sharpe Ratio",
    hovermode="x unified"
)

rolling_sharpe_chart.show()

Das Diagramm zeigt, wie stark die risikobereinigte Performance von AAPL über den Zeitraum schwankt – ein einzelner Gesamtwert würde diese Dynamik vollständig verbergen.

Drei Phasen stechen hervor:

- **2020 (COVID-Crash):** Die rollierende Sharpe Ratio sank spürbar, da der Kurseinbruch die rollierende Standardabweichung stark erhöhte. Da das 252-Tage-Fenster jedoch noch die außergewöhnlich starken Renditen aus 2019 umfasste, blieb die Kennzahl über 0. Die anschließende Erholung ließ die Sharpe Ratio danach stark ansteigen.
- **2021:** Die beste Phase – die Sharpe Ratio lag zeitweise deutlich über 1, was eine herausragende risikobereinigte Performance signalisiert.
- **2022 (Zinsanstieg):** Einbruch unter 0. Die rollierende 12-Monats-Rendite wurde negativ, da 2022 das einzige klar negative Kalenderjahr des Betrachtungszeitraums war. Da `RISK_FREE_RATE` im Code als fester Wert (2 %) definiert ist, spielen die tatsächlichen Fed-Zinserhöhungen für die Berechnung keine Rolle – ausschlaggebend war allein die negative rollierende Rendite.

Dieses Muster bestätigt: Die Sharpe Ratio ist keine stabile Kennzahl, sondern stark von Marktphasen abhängig. Für eine vollständige Risikobeurteilung ist die rollierende Betrachtung dem Gesamtwert klar vorzuziehen.

In [13]:
positive_days = (returns['ret'] > 0).sum()
negative_days = (returns['ret'] < 0).sum()

days_chart = px.pie(
    names=["Positive Handelstage", "Negative Handelstage"],
    values=[positive_days, negative_days],
    title="Apple Inc. (AAPL) – Anteil positiver vs. negativer Handelstage 2019–2024",
    color_discrete_sequence=["#00b09b", "#e74c3c"]
)

days_chart.update_traces(textinfo="percent+label")
days_chart.show()

Das Kreisdiagramm zeigt, dass AAPL im Betrachtungszeitraum an rund **54 % aller Handelstage** eine positive Tagesrendite verzeichnete. Diese leichte Mehrheit positiver Tage ist konsistent mit dem langfristigen Aufwärtstrend der Aktie und entspricht dem typischen Muster wachstumsstarker Titel. Entscheidend ist dabei nicht nur die Häufigkeit, sondern auch die Asymmetrie: Positive Handelstage überwiegen mengenmäßig, während die wenigen extremen negativen Tage (wie im März 2020) das Gesamtbild kurzfristig verzerren können.

### 4.4 Maximum Drawdown

Der **Maximum Drawdown (MDD)** misst den größtmöglichen kumulierten Verlust einer Anlage innerhalb eines bestimmten Zeitraums – konkret den prozentualen Rückgang vom höchsten Kurs (Peak) bis zum darauffolgenden tiefsten Kurs (Trough), bevor ein neues Hoch erreicht wird.

Er gibt Antwort auf die Frage: **Wie viel hätte ein Anleger im schlimmsten Fall verloren, wenn er zum ungünstigsten Zeitpunkt gekauft hätte?**

Die Formel lautet:

$$MDD = \frac{P_{trough} - P_{peak}}{P_{peak}}$$

| Variable | Bedeutung |
|---|---|
| $P_{peak}$ | Höchster Kurs (Peak) vor dem Einbruch |
| $P_{trough}$ | Tiefster Kurs (Trough) nach dem Peak |

**Interpretation:** Ein MDD von z. B. −30 % bedeutet, dass ein Anleger, der zum Höchstkurs eingestiegen wäre, zwischenzeitlich 30 % seines Einsatzes verloren hätte. Je kleiner der MDD (d. h. je näher an 0), desto besser die Risikokontrolle der Anlage.

Der MDD eignet sich besonders für:
- Risikoscheue Anleger, die maximale Verluste begrenzen möchten
- Institutionelle Investoren mit definierten Risikovorgaben
- Den Vergleich der Verlusttoleranz zwischen verschiedenen Anlagen

---

**Quelle:**
- [Maximum Drawdown – Fidelity Deutschland](https://www.fidelity.de/wissen/tipps-and-strategien/risiko-kennziffern/maximum-drawdown/)

In [14]:
drawdown = (aapl
            .sort_values('date')
            [['date', 'adjusted']]
            .assign(peak=lambda x: x['adjusted'].cummax()) # cummax() berechnet den kumulierten Maximalwert bis zum aktuellen Zeitpunkt
            .assign(drawdown=lambda x: (x['adjusted'] - x['peak']) / x['peak']) # MDD berechnen: (aktueller Kurs - Peak) / Peak
            )

max_drawdown = drawdown['drawdown'].min()
max_drawdown_date = drawdown.loc[drawdown['drawdown'].idxmin(), 'date']

print(f"Maximum Drawdown: {max_drawdown:.2%}") # % berechnet den Wert in Prozent um und .2 gibt an, dass 2 Nachkommastellen angezeigt werden sollen
print(f"Tiefpunkt erreicht am: {max_drawdown_date.date()}")

drawdown_chart = px.area(
    drawdown,
    x="date",
    y="drawdown",
    title="Apple Inc. (AAPL) – Drawdown 2019–2024",
    labels={"date": "Datum", "drawdown": "Drawdown"}
)

drawdown_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Drawdown",
    yaxis_tickformat=".0%",
    hovermode="x unified"
)

drawdown_chart.update_traces(fillcolor="rgba(231, 76, 60, 0.3)", line_color="#e74c3c")

drawdown_chart.show()

Maximum Drawdown: -31.43%
Tiefpunkt erreicht am: 2020-03-23


Das Diagramm zeigt den kumulierten Drawdown der AAPL-Aktie über den gesamten Betrachtungszeitraum 2019–2024. Die rote Fläche visualisiert zu jedem Zeitpunkt, wie weit der Kurs vom jeweils vorangegangenen Höchststand entfernt war – je tiefer die Fläche, desto größer der zwischenzeitliche Verlust.

Zwei Phasen stechen deutlich hervor:

- **COVID-Crash (Februar–März 2020):** Der tiefste Punkt des gesamten Zeitraums. Die Aktie fiel auf **–31,43 %** vom vorangegangenen Hoch – dies ist der **Maximum Drawdown** des Betrachtungszeitraums. Der Tiefpunkt wurde am **23. März 2020** erreicht, dem Tag des pandemiebedingten Börsenbodens in den USA.

- **Zinsanstiegsphase (Januar 2022 – Januar 2023):** Infolge der aggressiven Zinserhöhungen der US-Notenbank (Fed) – dem schnellsten Zinsanstieg seit den 1980er-Jahren – setzte ab dem Januarhoch 2022 (~182 USD) eine langanhaltende Korrektur ein. Steigende Zinsen erhöhen den Diskontierungssatz künftiger Gewinne und belasten damit besonders hoch bewertete Wachstumstitel wie Apple. Obwohl der Kurs im Sommer 2022 zwischenzeitlich etwas erholte, wurde das eigentliche Tief erst im **Januar 2023** bei rund 124 USD erreicht. Diese Phase erstreckte sich damit über ein volles Jahr und war deutlich langwieriger als der COVID-Schock.

Außerhalb dieser beiden Stressphasen blieb der Drawdown überwiegend gering. Die schnelle Erholung nach dem COVID-Schock – sichtbar an der raschen Rückkehr zur Nulllinie – steht in deutlichem Kontrast zur langsamen, schrittweisen Erholung nach dem 2022/2023-Einbruch.

---

**Quellen:**
- [Apple-Aktie: Kursverlust 2022 und Analysteneinschätzungen – Handelsblatt](https://www.handelsblatt.com/finanzen/maerkte/aktien/apple-aktie-24-prozent-kursverlust-seit-jahresanfang-lohnt-sich-jetzt-die-apple-aktie-das-sagen-analysten/28796758.html)
- [Apple's value plunged nearly $1 trillion in 2022 – ABC News](https://abcnews.go.com/Business/apples-plunged-1-trillion-2022-economy/story?id=95930467)

### 4.5 Durchschnittliche Rendite (Geometrisches Mittel)

Die **durchschnittliche Rendite** gibt an, wie stark eine Anlage im Durchschnitt pro Periode gewachsen ist. Es gibt zwei grundlegend unterschiedliche Berechnungsmethoden:

**Arithmetisches Mittel** – einfacher Durchschnitt aller Periodenrenditen:

$$r_{arith} = \frac{r_1 + r_2 + \cdots + r_n}{n}$$

**Geometrisches Mittel (CAGR)** – berücksichtigt den Zinseszinseffekt:

$$r_{geo} = \left(\prod_{t=1}^{n}(1 + r_t)\right)^{1/n} - 1$$

wobei das Produkt $\prod$ aller Faktoren $(1 + r_t)$ das tatsächliche Endvermögen aus einem Startwert von 1 ergibt, und die $n$-te Wurzel daraus die konstante jährliche Wachstumsrate liefert, die zum selben Ergebnis führt.

**Warum das geometrische Mittel?**

Das arithmetische Mittel überschätzt die tatsächliche Wertentwicklung systematisch, weil es Verluste und Gewinne so behandelt, als wären sie unabhängig voneinander. Fällt eine Aktie um 50 % und steigt danach um 100 %, ergibt das arithmetische Mittel **+25 %** – obwohl das Kapital unverändert geblieben ist. Das geometrische Mittel liefert korrekt **0 %**.

Der Unterschied zwischen beiden Werten wird als **Volatility Drag** bezeichnet: Je stärker die Renditen schwanken, desto größer die Lücke zwischen arithmetischem und geometrischem Mittel. Für eine ehrliche Aussage über die tatsächliche Vermögensentwicklung über mehrere Jahre ist das **geometrische Mittel die einzig korrekte Methode**.

| Methode | Berücksichtigt Zinseszins | Geeignet für |
|---|---|---|
| Arithmetisch | Nein | Sharpe Ratio, Erwartungswert-Schätzungen |
| Geometrisch | Ja | Langfristige Renditeangaben, Vergleiche |

---

**Quellen:**
- [Arithmetische, geometrische und logarithmische Rendite – anlegerplus.de](https://anlegerplus.de/renditeberechnungen-arithmetische-geometrische-logarithmische/)
- [Rendite richtig berechnen – Finanzfluss](https://www.finanzfluss.de/geldanlage/rendite-berechnen/)
- [Renditen und Renditetricks aufdecken – Finanzwesir](https://www.finanzwesir.com/blog/rendite-richtig-berechnen-renditetricks-aufdecken)

In [15]:
yearly_returns = (returns
                  .assign(year=lambda x: x['date'].dt.year)
                  .groupby('year')['ret']
                  .apply(lambda r: (1 + r).prod() - 1)
                  .reset_index()
                  .rename(columns={'ret': 'yearly_return'})
                  )

cagr = (returns['ret'] + 1).prod() ** (TRADING_DAYS_PER_YEAR / len(returns)) - 1
annual_arith_return = returns['ret'].mean() * TRADING_DAYS_PER_YEAR

print(f"Geometrische Rendite (CAGR):         {cagr:.2%}")
print(f"Arithmetische Rendite (annualisiert): {annual_arith_return:.2%}")
print(f"Volatility Drag:                      {annual_arith_return - cagr:.2%}")

yearly_returns_chart = px.bar(
    yearly_returns,
    x='year',
    y='yearly_return',
    title='Apple Inc. (AAPL) – Jährliche Rendite (geometrisch) 2019–2024',
    labels={'year': 'Jahr', 'yearly_return': 'Rendite'},
    color='yearly_return',
    color_continuous_scale=[[0, '#e74c3c'], [0.5, '#f39c12'], [1, '#00b09b']],
    color_continuous_midpoint=0
)

yearly_returns_chart.update_layout(
    xaxis_title='Jahr',
    yaxis_title='Rendite',
    yaxis_tickformat='.0%',
    coloraxis_showscale=False,
    xaxis=dict(tickmode='linear', dtick=1)
)

yearly_returns_chart.show()

Geometrische Rendite (CAGR):         37.38%
Arithmetische Rendite (annualisiert): 36.53%
Volatility Drag:                      -0.85%


Das Balkendiagramm zeigt die geometrische Jahresrendite für jedes Jahr des Betrachtungszeitraums. Die Farbe kodiert dabei das Vorzeichen: Grün steht für positive, Rot für negative Jahresrenditen.

Die Ergebnisse im Überblick:

- **2019 und 2020** verzeichneten trotz COVID-Crash starke positive Renditen – AAPL erholte sich so schnell, dass das Gesamtjahr 2020 dennoch deutlich positiv abschloss.
- **2021** war das stärkste Jahr des Zeitraums mit einer Rendite von über +25 %.
- **2022** war das einzige klar negative Jahr, getrieben durch den Zinsanstieg der Fed.
- **2023 und 2024** zeigten erneut starke Erholung.

Die **CAGR** (Compound Annual Growth Rate) gibt die konstante jährliche Wachstumsrate an, die über den gesamten Zeitraum zum tatsächlichen Endergebnis geführt hätte. Sie liegt spürbar unter der arithmetischen Rendite – die Differenz ist der **Volatility Drag**: die Dämpfung der Rendite durch Schwankungen, die das arithmetische Mittel ignoriert.

## 5. Fazit

Die Analyse der Apple-Aktie im Zeitraum 2019–2024 zeigt ein konsistentes Bild: AAPL ist eine Aktie mit überdurchschnittlicher Rendite bei gleichzeitig kontrollierbarem Risiko.

**Rendite:** Die geometrische Jahresrendite (CAGR) liegt deutlich im zweistelligen Bereich und spiegelt eine echte Vermögensvermehrung wider – trotz zwei gravierender Krisen im Betrachtungszeitraum. Besonders 2019, 2021 und 2023/2024 waren außergewöhnlich starke Jahre.

**Risiko:** Der Maximum Drawdown von –31,43 % (23. März 2020) zeigt, dass AAPL auch empfindlich auf externe Schocks reagiert. Die rollierende Volatilität macht deutlich, dass Marktstress in kurzen, intensiven Phasen auftritt – nicht als Dauerzustand. Außerhalb dieser Stressphasen war die Volatilität vergleichsweise gering.

**Risikobereingte Performance:** Die Sharpe Ratio bestätigt, dass AAPL seine Anleger für das eingegangene Risiko gut entschädigt hat. Eine Sharpe Ratio über 1 gilt als Zeichen herausragender risikobereinigter Performance.

**Krisenresilienz:** Beide Drawdown-Phasen wurden vollständig überwunden. Der COVID-Crash (2020) wurde innerhalb weniger Monate aufgeholt; die längere Korrekturphase 2022/2023 infolge der Fed-Zinserhöhungen endete ebenfalls in einer starken Erholung.

Zusammenfassend bestätigt die Analyse die eingangs formulierte Hypothese: Apple Inc. vereint langfristiges Wachstumspotenzial mit einer vergleichsweise hohen Widerstandsfähigkeit gegenüber Marktturbulenzen – und ist damit ein geeignetes Beispiel für die Anwendung der hier behandelten Finanzanalysemethoden.